In [1]:
import numpy as np 
import pandas as pd 

from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('CO2_Emissions_Canada.csv')
df.sample(5)

,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
5702,DODGE,Challenger (MDS),MID-SIZE,6.4,8,A8,Z,15.8,9.6,13.0,22,304
1116,AUDI,Q3 QUATTRO,COMPACT,2.0,4,AS6,Z,11.9,8.4,10.3,27,237
855,NISSAN,PATHFINDER,SUV - SMALL,3.5,6,AV,X,12.0,8.9,10.6,27,244
601,JEEP,WRANGLER UNLIMITED 4X4 (4-DOOR),SUV - SMALL,3.6,6,M6,X,15.0,11.4,13.4,21,308
1641,JAGUAR,F-TYPE COUPE,TWO-SEATER,3.0,6,AS8,Z,11.8,8.4,10.3,27,237


In [3]:
cat_cols = ['Make',	'Model', 'Vehicle Class']

for col in cat_cols:
    df[col] = df[col].str.lower()

df = df.drop_duplicates()

x = df.drop(columns=['CO2 Emissions(g/km)'])
y = df['CO2 Emissions(g/km)']
x_train, x_test, y_train, y_test = train_test_split(x, y , test_size=.2, random_state=42)

In [4]:
!pip install optuna

In [5]:
!pip install mlflow dagshub

In [6]:
import dagshub
dagshub.init(
    repo_owner='anni0955', 
    repo_name='CO2-Emission', 
    mlflow=True
)

Accessing as anni0955

Initialized MLflow to track repo "anni0955/CO2-Emission"

Repository anni0955/CO2-Emission initialized!

In [24]:
import mlflow 
mlflow.set_experiment('EXP 3 (Part 2)- OE Fuel Type + Log transformation')

2026/06/15 12:09:43 INFO mlflow.tracking.fluent: Experiment with name 'EXP 3 (Part 2)- OE Fuel Type + Log transformation' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/9bb1d19ed1704d489b9821f65f93a194', creation_time=1781525386576, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1781525386576, lifecycle_stage='active', name='EXP 3 (Part 2)- OE Fuel Type + Log transformation', tags={}, trace_location=None, workspace='default'>

In [25]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

import optuna

In [26]:
df.sample(5)

,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
5744,ford,escape awd,suv - small,1.5,4,AS6,X,11.2,8.4,9.9,29,232
6155,mercedes-benz,amg c 63,compact,4.0,8,A9,Z,13.2,8.7,11.2,25,263
2850,kia,optima hybrid ex,mid-size,2.4,4,AM6,X,6.7,6.1,6.4,44,150
2328,bmw,m4 coupe,compact,3.0,6,AM7,Z,14.1,9.7,12.1,23,284
238,chevrolet,silverado,pickup truck - standard,4.3,6,A6,E,18.9,14.2,16.8,17,269


In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import StandardScaler, PowerTransformer

In [30]:
transformer = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Make', 'Model', 'Vehicle Class', 'Transmission']),
    ('oe', OrdinalEncoder(categories=[['N', 'X', 'D', 'Z', 'E']]), ['Fuel Type']),
    ('pt', PowerTransformer(), ['Fuel Consumption City (L/100 km)',	'Fuel Consumption Hwy (L/100 km)',	'Fuel Consumption Comb (L/100 km)',	'Fuel Consumption Comb (mpg)']),
    ('scale', StandardScaler(),['Engine Size(L)', 'Cylinders', 'Fuel Consumption City (L/100 km)', 'Fuel Consumption Hwy (L/100 km)', 'Fuel Consumption Comb (L/100 km)', 'Fuel Consumption Comb (mpg)'])
])

In [31]:
x_train_trans = transformer.fit_transform(x_train)
x_test_trans = transformer.transform(x_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [32]:
x_train_trans.shape

(4792, 1582)

In [33]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [ ]:
def objective(trial):
    with mlflow.start_run(nested=True):
        model_name = trial.suggest_categorical('model', ['LR', 'SVR','DT', 'RF', 'GB', 'XGB'])

        if model_name == 'LR':
            model = LinearRegression()

        
        elif model_name == 'SVR':
            kernel_svm = trial.suggest_categorical('kernel_svm', ['linear', 'poly', 'rbf'])

            if kernel_svm == 'linear':
                c_linear = trial.suggest_float('c_linear', 1e-3, 10, log=True)
                model = SVR(C=c_linear, kernel='linear')

            elif kernel_svm == 'poly':
                c_poly = trial.suggest_float('c_poly', 1e-3, 10, log=True)
                degree_poly = trial.suggest_int('degree_poly', 1, 4)
                model = SVR(C=c_poly, degree=degree_poly, kernel='poly')

            else:
                c_rbf = trial.suggest_float('c_rbf', 1e-3, 10, log=True)
                gamma_rbf = trial.suggest_float('gamma_rbf', 1e-3, 10, log=True)
                model = SVR(C=c_rbf, gamma=gamma_rbf, kernel='rbf')

        elif model_name == 'DT':
            criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'absolute_error', 'poisson', 'squared_error'])
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = DecisionTreeRegressor(criterion=criterion, max_depth=max_depth)

        elif model_name == 'RF':
            n_estimators = trial.suggest_int('n_estimators', 50, 500)
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
        
        elif model_name == 'GB':
            n_estimators = trial.suggest_int('n_estimators', 50, 500)
            learning_rate = trial.suggest_float('learning_rate', 1e-2, 1)
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = GradientBoostingRegressor(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)

        else:
            n_estimators = trial.suggest_int('n_estimators', 50, 500)
            learning_rate = trial.suggest_float('learning_rate', 1e-2, 1)
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = XGBRegressor(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42, n_jobs=-1)    

        
        model.fit(x_train_trans, y_train)
        mlflow.log_params(trial.params)

        test_y_pred = model.predict(x_test_trans)
        train_y_pred = model.predict(x_train_trans)

        test_MAE = mean_absolute_error(y_test, test_y_pred)
        train_MAE = mean_absolute_error(y_train, train_y_pred)

        test_RMSE = root_mean_squared_error(y_test, test_y_pred)
        train_RMSE = root_mean_squared_error(y_train, train_y_pred)

        mlflow.log_metric('train_MAE', train_MAE)
        mlflow.log_metric('test_MAE', test_MAE)

        mlflow.log_metric('train_RMSE', train_RMSE)
        mlflow.log_metric('test_RMSE', test_RMSE)

        return test_RMSE


In [ ]:
study = optuna.create_study(direction='minimize', study_name='Model Selection')

with mlflow.start_run(run_name='Best Model') as parent:
    study.optimize(objective, n_trials=50, n_jobs=-1)
    mlflow.log_params(study.best_params)

    mlflow.log_metric('best_score', study.best_value)

[I 2026-06-15 12:13:22,260] A new study created in memory with name: Model Selection
[I 2026-06-15 12:13:39,271] Trial 1 finished with value: 20.319589226791987 and parameters: {'model': 'SVR', 'kernel_svm': 'linear', 'c_linear': 0.0070256312163760985}. Best is trial 1 with value: 20.319589226791987.


🏃 View run handsome-crab-404 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3/runs/16d55bf3430e49829cde4f7a9e1f2f0d
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3


[I 2026-06-15 12:13:41,520] Trial 2 finished with value: 4.892590391730232 and parameters: {'model': 'DT', 'criterion': 'poisson', 'max_depth': 15}. Best is trial 2 with value: 4.892590391730232.


🏃 View run serious-fawn-663 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3/runs/9d0b85ba5b00440d90e8a3d59a3d7b59
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3


[I 2026-06-15 12:15:03,197] Trial 3 finished with value: 4.268830600486034 and parameters: {'model': 'RF', 'n_estimators': 241, 'max_depth': 14}. Best is trial 3 with value: 4.268830600486034.


...

[I 2026-06-15 12:30:22,406] Trial 44 finished with value: 4.210643965466577 and parameters: {'model': 'RF', 'n_estimators': 365, 'max_depth': 15}. Best is trial 26 with value: 4.207516565205853.


🏃 View run bittersweet-goat-943 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3/runs/c7b386af229b4d3f8f9f33728d6b4749
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3
🏃 View run amusing-penguin-651 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3/runs/74edd173f5644f11884e7fa118eea5fb
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3
🏃 View run secretive-gull-209 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3/runs/46be4d53822f4055a3420c5251fe117f
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3


[I 2026-06-15 12:30:40,489] Trial 45 finished with value: 4.95468807220459 and parameters: {'model': 'XGB', 'n_estimators': 385, 'learning_rate': 0.9854971120558156, 'max_depth': 15}. Best is trial 26 with value: 4.207516565205853.
[I 2026-06-15 12:30:46,647] Trial 46 finished with value: 4.571562767028809 and parameters: {'model': 'XGB', 'n_estimators': 388, 'learning_rate': 0.526010071106499, 'max_depth': 15}. Best is trial 26 with value: 4.207516565205853.


🏃 View run Best Model at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3/runs/a7f9bb9a54084654ab94b065d2b87db0
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/3


KeyboardInterrupt: 